<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/model-risk/lessons/P04-L05-calibration-stability-depth/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/model-risk/lessons/P04-L05-calibration-stability-depth/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/model-risk/lessons/P04-L05-calibration-stability-depth/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/model-risk/lessons/P04-L05-calibration-stability-depth/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P04-L05 · Calibration and stability, at depth

**You will build:** equal-frequency bins that survive a score full of ties, the ECE
sensitivity table for one fixed model, a chi-square tail and the Hosmer-Lemeshow test with
the degrees of freedom its setting demands, a characteristic stability index that names the
input that moved, and a bootstrap of PSI under no drift at all — so that every threshold in
the appendix you generate carries the false-alarm rate it actually has at your sample size.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** T00-L01 (the tier gate and the profiler) and P04-L01 (the validation
suite, whose reliability table, ECE and PSI this lesson carries forward unchanged). Pure
numpy: there is no scipy here, so the chi-square tail is yours to build.

The data is **synthetic and generated in this notebook**. Every figure you see is computed
by code you run, including every figure in the appendix.

By the end you will be able to:

1. Implement equal-frequency calibration bins with a written rule for tied scores, and
   measure how far one model's ECE moves with the binning scheme and the bin count.
2. Implement the chi-square survival function and the Hosmer-Lemeshow test, and explain why
   its degrees of freedom depend on where the model was fitted.
3. Implement a characteristic stability index that attributes a population shift to the
   input that moved.
4. Measure, by bootstrapping PSI under no drift, the false-alarm rate a threshold carries at
   a given sample size.
5. Generate a calibration-and-stability appendix in which every threshold carries that rate.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import sys
import time
import traceback
from typing import Any, Callable, Mapping, NamedTuple

import numpy as np

SEED = 20260925
N_DEV = 8000            # development sample: the baseline every stability figure is cut on
N_VAL = 4000            # the validator's independent holdout, drawn from the same period
N_MON = 2000            # one monitoring month
N_BINS = 10             # module 1's default bin count
BIN_COUNTS = (5, 10, 15, 20, 30, 50)    # the bin counts the sensitivity table sweeps
HL_GROUPS = 10          # the customary number of Hosmer-Lemeshow groups
ALPHA = 0.05            # significance level for the goodness-of-fit test: a policy input
ECE_CEILING = 0.04      # module 1's promotion policy ceiling on ECE: a policy input
PSI_THRESHOLDS = (0.10, 0.25)           # the industry's conventional amber and red bands
PSI_FLOOR = 1e-6        # module 1's floor on a bin's share
N_BOOT = 1000           # bootstrap replicates behind every null distribution
SIZES = (50, 100, 200, 500, 1000, N_MON)   # monitoring sizes for the false-alarm table
N_SIMS = 400            # simulated samples in each goodness-of-fit study
N_ECE_SIMS = 200        # simulated outcome vectors behind each ECE noise floor

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__)

DATA_NOTE = (
    "SYNTHETIC DATA. Every record in this notebook was generated inside it by "
    f"numpy.random.default_rng({SEED}). No real applicant, account or lending decision is "
    "represented. The model under validation, its master scale and the planted shifts in the "
    "monitoring months are all written out in the setup cell, so every finding can be checked "
    "against the world that produced it."
)

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("equal_frequency_edges",),
    "exercise 2": ("ece_on_edges",),
    "exercise 3": ("ece_sensitivity",),
    "exercise 4": ("chi2_sf",),
    "exercise 5": ("hosmer_lemeshow",),
    "exercise 6": ("characteristic_edges", "characteristic_stability"),
    "exercise 7": ("psi_null_bootstrap",),
    "exercise 8": ("false_alarm_rate", "threshold_for_rate"),
    "exercise 9": ("render_appendix",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (ece_sensitivity)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


# --- Module 1, carried forward. Faithful copies: same names, same conventions. -------------

class ReliabilityTable(NamedTuple):
    """One row per bin, empty bins included. Given to you: module 1's record, unchanged."""

    lo: np.ndarray          # lower edge of each bin
    hi: np.ndarray          # upper edge of each bin
    count: np.ndarray       # number of records in the bin, integer dtype
    mean_pred: np.ndarray   # mean predicted probability in the bin, nan when empty
    obs_rate: np.ndarray    # observed event rate in the bin, nan when empty


def reliability_table(y_true: np.ndarray, y_prob: np.ndarray,
                      n_bins: int = N_BINS) -> ReliabilityTable:
    """Module 1's reliability table. Given to you.

    Equal-width bins over [0, 1]: bin m covers ((m-1)/M, m/M], a probability on an interior
    edge belongs to the LOWER bin, 0.0 belongs to the first bin, and an empty bin reports a
    count of 0 and nan for both rates.
    """
    y = np.asarray(y_true)
    p = np.asarray(y_prob, dtype=float)
    if n_bins < 1:
        raise ValueError(f"n_bins must be at least 1, got {n_bins}")
    if y.shape != p.shape:
        raise ValueError(f"y_true and y_prob differ in shape: {y.shape} vs {p.shape}")
    if p.size and (np.nanmin(p) < 0.0 or np.nanmax(p) > 1.0 or not np.all(np.isfinite(p))):
        raise ValueError("y_prob must hold finite probabilities inside [0, 1]")
    if not np.all(np.isin(y, (0, 1))):
        raise ValueError("y_true must hold only 0 and 1")
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.searchsorted(edges, p, side="left") - 1, 0, n_bins - 1)
    count = np.bincount(idx, minlength=n_bins).astype(np.int64)
    pred_sum = np.bincount(idx, weights=p, minlength=n_bins)
    event_sum = np.bincount(idx, weights=y.astype(float), minlength=n_bins)
    with np.errstate(invalid="ignore", divide="ignore"):
        denom = np.where(count > 0, count, np.nan)
        mean_pred = pred_sum / denom
        obs_rate = event_sum / denom
    return ReliabilityTable(lo=edges[:-1], hi=edges[1:], count=count,
                            mean_pred=mean_pred, obs_rate=obs_rate)


def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray,
                               n_bins: int = N_BINS) -> float:
    """Module 1's ECE: the support-weighted mean absolute gap over the reliability table.

    Given to you. Empty bins contribute nothing, each bin weighs count / total, the gap is
    absolute, and the result is a plain Python float.
    """
    table = reliability_table(y_true, y_prob, n_bins)
    total = int(table.count.sum())
    if total == 0:
        raise ValueError("expected calibration error is undefined on an empty sample")
    filled = table.count > 0
    weights = table.count[filled] / total
    gaps = np.abs(table.obs_rate[filled] - table.mean_pred[filled])
    return float(np.sum(weights * gaps))


class StabilityResult(NamedTuple):
    """PSI and the per-bin arithmetic that produced it. Given to you: module 1's record."""

    psi: float                   # the total
    contributions: np.ndarray    # per-bin (a - e) * ln(a / e), same length as the bins
    expected_pct: np.ndarray     # baseline share per bin, summing to 1.0
    actual_pct: np.ndarray       # current share per bin, summing to 1.0


def population_stability_index(expected: np.ndarray, actual: np.ndarray,
                               edges: np.ndarray, floor: float = PSI_FLOOR) -> StabilityResult:
    """Module 1's PSI of `actual` against baseline `expected`, on edges cut ONCE on the baseline.

    Given to you. Bin i is [edges[i], edges[i+1]), the last bin is closed on the right, values
    outside the edges fall into the nearest end bin, shares are taken over each side's own
    total, and every share is floored at `floor` on BOTH sides before the logarithm.
    """
    edges = np.asarray(edges, dtype=float)
    if edges.size < 2:
        raise ValueError(f"edges needs at least two values, got {edges.size}")
    if not np.all(np.diff(edges) > 0):
        raise ValueError("edges must be strictly increasing")
    n_bins = edges.size - 1

    def _shares(x: np.ndarray) -> np.ndarray:
        v = np.asarray(x, dtype=float)
        if v.size == 0:
            raise ValueError("population_stability_index needs a non-empty sample on both sides")
        idx = np.clip(np.searchsorted(edges, v, side="right") - 1, 0, n_bins - 1)
        return np.bincount(idx, minlength=n_bins).astype(float) / v.size

    e_pct = _shares(expected)
    a_pct = _shares(actual)
    e_safe = np.maximum(e_pct, floor)
    a_safe = np.maximum(a_pct, floor)
    contributions = (a_safe - e_safe) * np.log(a_safe / e_safe)
    return StabilityResult(psi=float(contributions.sum()), contributions=contributions,
                           expected_pct=e_pct, actual_pct=a_pct)


def quantile_edges(x: np.ndarray, n_bins: int = N_BINS) -> np.ndarray:
    """Module 1's stability edges. Given to you, and section 7 shows where they break.

    Linear-interpolated quantiles of `x`, de-duplicated, with the outer edges opened to
    infinity so nothing falls outside them.
    """
    edges = np.quantile(np.asarray(x, dtype=float), np.linspace(0.0, 1.0, n_bins + 1))
    edges = np.unique(edges)
    edges[0] = -np.inf
    edges[-1] = np.inf
    return edges


# --- The world, and the model under validation. SYNTHETIC — see DATA_NOTE. ----------------

DRIVERS = ("channel_online", "delinquencies", "enquiries", "months_on_book", "utilisation")

# How the outcome really depends on the drivers, on the log-odds scale.
TRUE_LOG_ODDS = {"intercept": -3.35, "channel_online": 0.04, "delinquencies": 0.62,
                 "enquiries": 0.21, "months_on_book": -0.0045, "utilisation": 2.9}
# The model under validation: fitted by the first line on an earlier book, and now fixed. It
# leans too hard on utilisation and sits a little low everywhere else.
MODEL_LOG_ODDS = {"intercept": -3.60, "channel_online": 0.05, "delinquencies": 0.55,
                  "enquiries": 0.19, "months_on_book": -0.0040, "utilisation": 3.5}
# The model does not publish its raw probability. It publishes the PD of the grade that
# probability falls in, on a twenty-grade master scale — which is how most rating systems
# report, and why most of the scores in this notebook are tied.
MASTER_SCALE_UPPER = np.array([0.0030, 0.0039, 0.0051, 0.0067, 0.0088, 0.0115, 0.0150,
                               0.0200, 0.0260, 0.0340, 0.0440, 0.0580, 0.0760, 0.1000,
                               0.1300, 0.1700, 0.2300, 0.3000, 0.4500])
MASTER_SCALE_PD = np.array([0.0020, 0.0034, 0.0045, 0.0059, 0.0077, 0.0100, 0.0130, 0.0175,
                            0.0230, 0.0300, 0.0390, 0.0510, 0.0670, 0.0880, 0.1150, 0.1500,
                            0.2000, 0.2600, 0.3700, 0.5500])


def _log_odds(drivers: Mapping[str, np.ndarray], coefficients: Mapping[str, float]) -> np.ndarray:
    out = np.full(drivers[DRIVERS[0]].shape, float(coefficients["intercept"]))
    for name in DRIVERS:
        out = out + coefficients[name] * drivers[name]
    return out


def synthetic_book(rng: np.random.Generator, n: int, online_share: float = 0.30,
                   utilisation_shift: float = 0.0) -> dict:
    """One deterministic synthetic book of applications. SYNTHETIC — see DATA_NOTE.

    `online_share` and `utilisation_shift` are the two levers the monitoring months pull. The
    model itself never changes: only the population it is applied to does.
    """
    drivers = {
        "channel_online": (rng.random(n) < online_share).astype(float),
        "delinquencies": np.minimum(rng.poisson(0.35, n), 6).astype(float),
        "enquiries": np.minimum(rng.poisson(1.2, n), 10).astype(float),
        "months_on_book": rng.integers(6, 181, n).astype(float),
        "utilisation": np.clip(rng.beta(2.0, 3.0, n) + utilisation_shift, 0.0, 1.0),
    }
    y = (rng.random(n) < _sigmoid(_log_odds(drivers, TRUE_LOG_ODDS))).astype(np.int64)
    model = _log_odds(drivers, MODEL_LOG_ODDS)
    grade = np.searchsorted(MASTER_SCALE_UPPER, _sigmoid(model), side="left")
    return {"drivers": drivers, "y": y, "log_odds": model, "pd": MASTER_SCALE_PD[grade]}


_rng = np.random.default_rng(SEED)
DEV = synthetic_book(_rng, N_DEV)
VAL = synthetic_book(_rng, N_VAL)
MON_A = synthetic_book(_rng, N_MON, online_share=0.70)          # the online month
MON_B = synthetic_book(_rng, N_MON, utilisation_shift=0.12)     # the utilisation month
for _name, _book in (("development", DEV), ("holdout", VAL), ("month A", MON_A),
                     ("month B", MON_B)):
    print(f"{_name + ' sample:':<21} {_book['y'].size:>5} records, event rate "
          f"{_book['y'].mean():.4f}, mean published PD {_book['pd'].mean():.4f}")
print("\n" + DATA_NOTE)

## 1. The phenomenon: one model, several calibration errors

Module 1 fixed a binning scheme — ten equal-width bins — so that its exercise had one right
answer. A validator does not get that luxury. The first line picked its bins, the vendor's
tool picked different ones, and the committee will be told a single number.

The widely cited statement of the expected calibration error, in Guo and colleagues' paper,
partitions the predictions into `M` equally spaced bins and leaves `M` as a parameter; their
own tables use fifteen, module 1 used ten. Does it matter? Run this: it is module 1's own
function, unchanged, on one fixed model and one fixed holdout, with nothing varied but the
bin count.

In [ ]:
_distinct, _counts = np.unique(VAL["pd"], return_counts=True)
print(f"the model publishes {MASTER_SCALE_PD.size} grade PDs; {_distinct.size} of them occur "
      f"in the holdout, and the most common grade holds {_counts.max() / _counts.sum():.1%} "
      "of the records")
for _m in (5, 10, 20, 50):
    print(f"module 1's ECE with {_m:>2} equal-width bins: "
          f"{expected_calibration_error(VAL['y'], VAL['pd'], _m):.4f}")
print("\nSame model, same records, same function. Only the bin count moved.")

## 2. Exercise 1 — `equal_frequency_edges()`

Equal-width bins put the edges where the ruler says. Equal-frequency bins put them where the
records are: each bin is meant to hold the same share of the sample, so every row of a
reliability table carries the same weight of evidence. It is also how the Hosmer-Lemeshow
test in exercise 5 forms its groups.

The trap is the master scale. When a large share of the book sits on one published PD,
the quantile edges meant to fall inside that share all land on the same value. There are three
things you could do, and only one is defensible:

- **Split the tie.** Send some records on one grade's PD to one bin and the rest to the next.
  The counts come out equal — and two identical scores are treated differently depending on
  the order the extract happened to be sorted in. Rejected.
- **Keep the empty bins.** Report the bin count you asked for, some bins holding nothing. A
  bin between two coincident edges is not "no observations up there"; it is not a bin.
- **Merge coincident edges, and report how many bins you actually got.** This is the rule.
  It never splits a tie, never puts an edge at a score the model did not publish, never
  leaves a bin empty — and the number of bins it returns is part of the result.

The bins stay module 1's: `[0, e1]`, then `(e1, e2]`, and so on, closed on the right and
spanning `[0, 1]`, with `0.0` in the first bin.

<details><summary>💡 Hint 1 — what to think about</summary>

Sort the scores and ask which record closes the first share of them, which closes the
second, and so on. The SCORE of each such record is an edge. When several shares close on
the same score, those edges are one edge. The two outer edges are not scores at all: they are
the ends of the probability scale.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate first: at least one bin, a non-empty sample, every probability finite and inside
[0, 1]. Sort. For each k from 1 to n_bins, find the record that closes the first k shares by
integer arithmetic — its rank is the ceiling of k times the sample size over n_bins — and take
its score. De-duplicate those scores. The last one is the largest score in the sample: swap it
for 1.0, put 0.0 in front, and de-duplicate once more in case the smallest score was 0.0.

</details>

In [ ]:
def equal_frequency_edges(y_prob: np.ndarray, n_bins: int = N_BINS) -> np.ndarray:
    """Calibration bin edges holding equal shares of the records, ties never split.

    Rule, and every clause of it is graded:
      * sort the probabilities; for k = 1 .. n_bins, the k-th candidate edge is the score of
        the record at rank ceil(k * n / n_bins), counting ranks from 1 — the record that
        closes the first k shares. Use integer arithmetic for the rank.
      * candidate edges that coincide are merged, so on a tied score you get FEWER bins than
        you asked for. That is the answer, not an error.
      * the last candidate is the largest score; it is replaced by 1.0, and 0.0 is put in
        front, so the edges span [0, 1] as module 1's did. Merge again if the smallest score
        was itself 0.0.
      * the result is strictly increasing, every interior edge is a score that occurs in the
        sample, and on the sample it was cut from no bin (module 1's right-closed bins) is
        empty.
      * `ValueError` if `n_bins < 1`, if the sample is empty, or if any probability is not
        finite or lies outside [0, 1].

    Returns: the edges — a strictly increasing float array from 0.0 to 1.0, one element longer
    than the number of bins actually formed.

    Example:
        >>> equal_frequency_edges(np.array([0.1, 0.2, 0.3, 0.4]), 2)
        array([0. , 0.2, 1. ])
        >>> equal_frequency_edges(np.array([0.1] * 6 + [0.2, 0.3, 0.4, 0.5]), 5)  # 3 bins, not 5
        array([0. , 0.1, 0.3, 1. ])
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_equal_frequency_edges() -> None:
    plain = equal_frequency_edges(np.array([0.1, 0.2, 0.3, 0.4]), 2)
    assert isinstance(plain, np.ndarray), "return a numpy array of edges"
    assert plain.tolist() == [0.0, 0.2, 1.0], (
        f"four untied scores in two bins gave edges {plain.tolist()}, not [0.0, 0.2, 1.0]. The "
        "second record closes the first half, so its score, 0.2, is the edge. 0.25 means you "
        "used np.quantile's default linear method, which invents a score nobody was given"
    )
    tied = equal_frequency_edges(np.array([0.1] * 6 + [0.2, 0.3, 0.4, 0.5]), 5)
    assert np.all(np.diff(tied) > 0), (
        f"edges {tied.tolist()} are not strictly increasing — coincident edges must be merged, "
        "not kept as empty bins"
    )
    assert tied.tolist() == [0.0, 0.1, 0.3, 1.0], (
        f"six records tied at 0.1 and five bins requested gave {tied.tolist()}; expected "
        "[0.0, 0.1, 0.3, 1.0] — three bins, because the first three shares all close on 0.1. "
        "Splitting the tie to force five bins is the mistake this rule exists to prevent"
    )
    assert plain[0] == 0.0 and plain[-1] == 1.0, "the edges must span [0, 1], as module 1's did"
    for bad, why in (((np.array([0.2, 0.4]), 0), "n_bins of 0"),
                     ((np.array([0.2, 1.4]), 2), "a probability above 1"),
                     ((np.array([]), 2), "an empty sample")):
        try:
            equal_frequency_edges(*bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"{why} should raise ValueError, not be quietly accepted")
    print("exercise 1 looks right — coincident edges merge and no tie is split")

In [ ]:
_try("exercise 1", _check_equal_frequency_edges)

Now point it at the model. The published PD is tied into the master scale's grades; the
model's raw probability — the one it computed before grading — is not. Same records, same
rule, very different answers to "how many bins did you get?"

In [ ]:
def _show_tied_bins() -> None:
    raw = _sigmoid(VAL["log_odds"])
    print(f"{'bins requested':>14}  {'published PD':>12}  {'raw probability':>15}")
    for m in BIN_COUNTS:
        got_pd = equal_frequency_edges(VAL["pd"], m).size - 1
        got_raw = equal_frequency_edges(raw, m).size - 1
        print(f"{m:>14}  {got_pd:>12}  {got_raw:>15}")
    print("\nA table that says '50 equal-frequency bins' over the published PD is describing a")
    print("table that does not exist. The number of bins used is part of the result.")


_try("tied bins", _show_tied_bins, needs=("exercise 1",))

## 3. Exercise 2 — `ece_on_edges()`

Module 1's ECE took a bin COUNT and built equal-width edges itself. This one takes the
EDGES, so the same reduction can run over any partition of `[0, 1]`: equal-width,
equal-frequency, or whatever a vendor's tool used. Everything else is carried forward
exactly — right-closed bins with `0.0` in the first, empty bins contributing nothing, each
bin weighted by its share of the records, the gap taken as an absolute value, a plain float
back. Given equal-width edges it must reproduce module 1's figure to the last digit, and the
check below holds it to that.

<details><summary>💡 Hint 1 — what to think about</summary>

Nothing about the reduction changes; only where the bins come from. Which bin does a
probability that sits exactly on an interior edge belong to, under module 1's convention?
And notice what a bin's weighted gap simplifies to: its share of the records times the
absolute gap between its mean outcome and its mean prediction.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Check the edges (strictly increasing, starting at 0.0 and ending at 1.0) and the arrays
(same length, non-empty, probabilities in [0, 1], labels 0 or 1). Place each probability with
a sorted search that sends a value on an edge to the left, step back one, and clip into range.
Per bin, count the records and sum the predictions and the outcomes. Over the bins that hold
records, add up the absolute difference between the outcome sum and the prediction sum, and
divide by the number of records. Return it as a Python float.

</details>

In [ ]:
def ece_on_edges(y_true: np.ndarray, y_prob: np.ndarray, edges: np.ndarray) -> float:
    """Module 1's expected calibration error, over any bin edges spanning [0, 1].

    Requirements, each graded:
      * bins are module 1's: [edges[0], edges[1]], then (edges[i], edges[i+1]] — closed on
        the RIGHT, so a probability on an interior edge belongs to the LOWER bin, and 0.0
        belongs to the first bin.
      * empty bins contribute nothing; each occupied bin weighs count / total; the gap is the
        ABSOLUTE difference between its observed rate and its mean prediction.
      * with `edges = np.linspace(0, 1, M + 1)` the result equals module 1's
        `expected_calibration_error(y_true, y_prob, M)`.
      * the return type is a plain Python `float`.
      * `ValueError` if the edges are not strictly increasing, do not start at 0.0 and end at
        1.0, or number fewer than two; if the arrays differ in length or are empty; if a
        probability lies outside [0, 1]; or if a label is not 0 or 1.

    Returns: the expected calibration error over those edges, as a plain Python float.

    Example — one event predicted at 0.1 sits on the edge and joins the lower bin:
        >>> ece_on_edges(np.array([1, 0]), np.array([0.1, 0.3]), np.array([0.0, 0.1, 1.0]))
        0.6
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_ece_on_edges() -> None:
    worked = ece_on_edges(np.array([1, 0]), np.array([0.1, 0.3]), np.array([0.0, 0.1, 1.0]))
    assert abs(worked - 0.6) < 1e-12, (
        f"the docstring example gave {worked:.6f}, not 0.6. 0.3 means 0.1 went to the UPPER "
        "bin: bins are closed on the right, so search with side='left' and step back one"
    )
    assert type(worked) is float, (
        f"return a plain Python float, got {type(worked).__name__} — wrap the sum in float()"
    )
    for m in (4, 10):
        mine = ece_on_edges(VAL["y"], VAL["pd"], np.linspace(0.0, 1.0, m + 1))
        theirs = expected_calibration_error(VAL["y"], VAL["pd"], m)
        assert abs(mine - theirs) < 1e-12, (
            f"with {m} equal-width edges you got {mine:.6f} and module 1 got {theirs:.6f}. "
            "Same edges must give the same ECE: check the bin convention and that empty bins "
            "contribute nothing"
        )
    try:
        ece_on_edges(np.array([0, 1]), np.array([0.2, 0.8]), np.array([0.0, 0.5, 0.9]))
    except ValueError:
        pass
    else:
        raise AssertionError("edges ending at 0.9 leave part of [0, 1] unbinned — ValueError")
    print("exercise 2 looks right — and it agrees with module 1 wherever module 1 has an answer")

In [ ]:
_try("exercise 2", _check_ece_on_edges)

## 4. Exercise 3 — `ece_sensitivity()`

One model, one holdout, two schemes, six bin counts: twelve ECEs. The table is the
evidence a validator attaches when asked "is 0.02 good?" — because the honest answer is
"under which binning?" Each row carries what a reader needs to judge it: how many bins were
asked for, how many were used, how many held records, and how thin the thinnest one was.

<details><summary>💡 Hint 1 — what to think about</summary>

Two schemes, and they differ in where the edges come from, not in how ECE is reduced. For
equal-width, "bins used" is simply what you asked for — some may be empty. For
equal-frequency, it is whatever survived the merge. The thinnest bin is the smallest count
among the bins that hold anything. Row order is part of the contract.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Reject an empty list of bin counts, or any count below one. Then loop over the two scheme
names in order and, inside, over the bin counts in the order given. Build the edges — evenly
spaced over [0, 1] for equal-width, your exercise-1 function for equal-frequency — place the
records in bins exactly as exercise 2 does, count them, and fill one row: scheme, requested,
bins used, bins occupied, the smallest occupied count, and the ECE from exercise 2.

</details>

In [ ]:
SCHEMES = ("equal-width", "equal-frequency")


class SensitivityRow(NamedTuple):
    """One binning scheme's reading of the same model."""

    scheme: str          # "equal-width" | "equal-frequency"
    requested: int       # bins asked for
    bins_used: int       # bins the edges actually define
    occupied: int        # bins holding at least one record
    min_support: int     # records in the thinnest occupied bin
    ece: float           # ece_on_edges over those edges


def ece_sensitivity(y_true: np.ndarray, y_prob: np.ndarray,
                    bin_counts: tuple = BIN_COUNTS) -> tuple:
    """The ECE of ONE fixed set of predictions under every scheme and bin count.

    Requirements, each graded:
      * every equal-width row first, then every equal-frequency row; within a scheme, the bin
        counts in the order given.
      * equal-width edges are `np.linspace(0, 1, m + 1)`, so `bins_used == requested`;
        equal-frequency edges come from `equal_frequency_edges`, so `bins_used` is what
        survived the merge and may be smaller.
      * `occupied` counts bins holding at least one record under module 1's right-closed
        convention; `min_support` is the smallest count among those.
      * `ece` is `ece_on_edges` over the row's edges, a plain float; the integer fields are
        plain ints.
      * `ValueError` if `bin_counts` is empty or holds a count below 1.

    Returns: a tuple of SensitivityRow, one per scheme and bin count, in the order above.

    Example:
        >>> rows = ece_sensitivity(np.array([0, 1, 0, 1]), np.array([0.2, 0.2, 0.2, 0.9]), (2, 4))
        >>> [(r.scheme[6:], r.requested, r.bins_used) for r in rows]
        [('width', 2, 2), ('width', 4, 4), ('frequency', 2, 2), ('frequency', 4, 2)]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_ece_sensitivity() -> None:
    y = np.array([0, 1, 0, 1])
    p = np.array([0.2, 0.2, 0.2, 0.9])
    rows = ece_sensitivity(y, p, (2, 4))
    assert len(rows) == 4 and all(isinstance(r, SensitivityRow) for r in rows), (
        f"two schemes times two bin counts is four SensitivityRow rows, got {len(rows)}"
    )
    assert [(r.scheme, r.requested) for r in rows] == [
        ("equal-width", 2), ("equal-width", 4), ("equal-frequency", 2), ("equal-frequency", 4)], (
        "rows run through every equal-width count first, then every equal-frequency count, "
        "each in the order given"
    )
    assert rows[3].bins_used == 2, (
        f"four equal-frequency bins over three tied scores and one 0.9 define "
        f"{rows[3].bins_used} bins; the tie merges them to 2. Report the bins USED, not the "
        "bins requested"
    )
    assert rows[1].bins_used == 4 and rows[1].occupied == 2, (
        "four equal-width bins are four bins even when only two hold records: bins_used 4, "
        f"occupied 2 — got bins_used {rows[1].bins_used}, occupied {rows[1].occupied}"
    )
    assert rows[1].min_support == 1, (
        f"the thinnest OCCUPIED bin holds 1 record, got {rows[1].min_support} — an empty bin "
        "is not the thinnest bin, it is not a bin with evidence in it"
    )
    want = ece_on_edges(y, p, np.linspace(0.0, 1.0, 3))
    assert abs(rows[0].ece - want) < 1e-12 and type(rows[0].ece) is float, (
        "each row's ece is ece_on_edges over that row's edges, as a plain float"
    )
    try:
        ece_sensitivity(y, p, ())
    except ValueError:
        pass
    else:
        raise AssertionError("an empty list of bin counts should raise ValueError")
    print("exercise 3 looks right — every row says how many bins it really had")

In [ ]:
_try("exercise 3", _check_ece_sensitivity)

Here is the table for the model under validation. Read the ECE column top to bottom and ask
which of these numbers you would have put on the summary page.

In [ ]:
def _show_sensitivity() -> None:
    rows = ece_sensitivity(VAL["y"], VAL["pd"], BIN_COUNTS)
    print(f"{'scheme':<16}{'requested':>10}{'used':>6}{'occupied':>10}{'thinnest':>10}"
          f"{'ECE':>9}")
    for r in rows:
        print(f"{r.scheme:<16}{r.requested:>10}{r.bins_used:>6}{r.occupied:>10}"
              f"{r.min_support:>10}{r.ece:>9.4f}")
    eces = [r.ece for r in rows]
    lo = rows[int(np.argmin(eces))]
    hi = rows[int(np.argmax(eces))]
    print(f"\nthe same model's ECE runs from {lo.ece:.4f} ({lo.scheme}, {lo.requested} bins) "
          f"to {hi.ece:.4f} ({hi.scheme}, {hi.requested} bins)")
    print(f"the largest reading is {hi.ece / lo.ece:.2f} times the smallest; "
          f"the spread is {hi.ece - lo.ece:.4f}")


_try("sensitivity table", _show_sensitivity, needs=("exercise 3",))

That spread has two sources, and they pull in opposite directions. Coarse bins MERGE
grades whose errors have opposite signs, and the merged gaps cancel — the reading falls.
Fine bins cannot split a tied grade, but on an untied score they would chase noise, and a
gap measured on forty records is mostly noise — the reading rises.

So how much ECE does a model score when it is EXACTLY right? You can measure that: keep the
model's own predictions, draw the outcomes from them, and compute ECE under each scheme. A
model scoring no more than that is calibrated as far as this sample can tell.

In [ ]:
def exact_calibration_ece(y_prob: np.ndarray, edges: np.ndarray, n_sims: int,
                          rng: np.random.Generator) -> np.ndarray:
    """ECE of `n_sims` outcome vectors drawn from `y_prob` itself. Given to you.

    A model whose predictions ARE the outcome probabilities is calibrated by construction, so
    this is the ECE the binning scheme reports for nothing but sampling noise.
    """
    p = np.asarray(y_prob, dtype=float)
    return np.array([ece_on_edges((rng.random(p.size) < p).astype(np.int64), p, edges)
                     for _ in range(n_sims)])


def _show_noise_floor() -> None:
    rng = np.random.default_rng(SEED + 3)
    print(f"{'scheme':<16}{'requested':>10}{'observed':>10}{'exact: mean':>13}"
          f"{'exact: 95th':>13}  above the 95th?")
    above = 0
    rows = ece_sensitivity(VAL["y"], VAL["pd"], BIN_COUNTS)
    for r in rows:
        edges = (np.linspace(0.0, 1.0, r.requested + 1) if r.scheme == "equal-width"
                 else equal_frequency_edges(VAL["pd"], r.requested))
        floor = exact_calibration_ece(VAL["pd"], edges, N_ECE_SIMS, rng)
        p95 = float(np.quantile(floor, 0.95))
        above += r.ece > p95
        print(f"{r.scheme:<16}{r.requested:>10}{r.ece:>10.4f}{floor.mean():>13.4f}"
              f"{p95:>13.4f}  {'yes' if r.ece > p95 else 'no'}")
    print(f"\n{above} of {len(rows)} schemes put this model's ECE above what an exactly "
          "calibrated model scores 95% of the time.")
    print("The noise floor moves with the scheme too. An ECE without its floor is half a number.")


_try("noise floor", _show_noise_floor, needs=("exercise 1", "exercise 2", "exercise 3"))

## 5. Exercise 4 — `chi2_sf()`

ECE has no reference distribution. A grouped goodness-of-fit statistic does: it is compared
with a chi-square distribution, and its p-value is that distribution's upper tail — the
SURVIVAL function, `Q(x; k) = P(X > x)` for `k` degrees of freedom. There is no scipy here,
so you build it.

The NIST/SEMATECH handbook writes the chi-square CDF as the incomplete gamma function at
`(k/2, x/2)`, normalised; the NIST Digital Library of Mathematical Functions supplies
everything else. Normalising DLMF 8.8.2 gives a recurrence that climbs two degrees of freedom
at a time, and DLMF 8.4.5 and 8.4.6 give the two places to start:

- `Q(x; 1) = erfc(sqrt(x / 2))`, and `Q(x; 2) = exp(-x / 2)`;
- `Q(x; k + 2) = Q(x; k) + (x/2)^(k/2) · exp(-x/2) / Γ(k/2 + 1)`.

Every degree of freedom is an odd or an even climb from one of those two. The only care the
arithmetic needs is at large `k`, where `(x/2)^(k/2)` and `Γ(k/2 + 1)` each overflow a double
long before their ratio does — so form each term through its logarithm (`math.lgamma`).

<details><summary>💡 Hint 1 — what to think about</summary>

Decide the parity of k first: it picks the starting value and the first power of x/2 that
gets added. Then how many terms are there? Each term moves two degrees of freedom, so count
from the start to k. And what does a statistic of zero or less mean for an upper tail?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Reject a degrees-of-freedom value that is not a whole number of at least one (a bool is not
one), and a statistic that is not a number. Anything at or below zero has upper tail 1.0,
and positive infinity has 0.0. Halve x. Start from the complementary error function of its
square root with a half-integer exponent for odd k, or from its negative exponential with
exponent one for even k. While twice the exponent is below k, add the term — built as the
exponential of (exponent times the log of half x, minus half x, minus the log-gamma of
exponent plus one) — and raise the exponent by one. Cap at 1.0 and return a float.

</details>

In [ ]:
def chi2_sf(x: float, dof: int) -> float:
    """Upper tail P(X > x) of a chi-square variable with `dof` degrees of freedom.

    Requirements, each graded:
      * exact, not simulated or interpolated: agrees with the published critical values in
        the NIST/SEMATECH table to the table's three decimals, and with an independent
        incomplete-gamma computation to about 1e-10.
      * any whole number of degrees of freedom from 1 upwards — including a few hundred,
        where naive powers and factorials overflow.
      * `x <= 0` returns 1.0; `x = inf` returns 0.0; the result is a plain Python float.
      * `ValueError` if `dof` is not a whole number of at least 1 (a bool is not accepted),
        or if `x` is nan.

    Returns: P(X > x), a plain Python float between 0.0 and 1.0.

    Example:
        >>> round(chi2_sf(2.0, 2), 6)       # two degrees of freedom: exactly exp(-x / 2)
        0.367879
        >>> round(chi2_sf(3.841, 1), 3)     # the 5% critical value for one degree of freedom
        0.05
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_chi2_sf() -> None:
    two = chi2_sf(2.0, 2)
    assert abs(two - math.exp(-1.0)) < 1e-12, (
        f"chi2_sf(2, 2) gave {two:.6f}; with two degrees of freedom the tail is exactly "
        "exp(-x / 2) = 0.367879. If you got 0.632121 you returned the CDF, not the upper tail"
    )
    assert type(two) is float, f"return a plain Python float, got {type(two).__name__}"
    # Upper-tail 5% critical values, NIST/SEMATECH e-Handbook table 1.3.6.7.4 (claims.yaml).
    for x, k in ((3.841, 1), (5.991, 2), (15.507, 8), (18.307, 10), (124.342, 100)):
        got = chi2_sf(x, k)
        assert abs(got - 0.05) < 2e-4, (
            f"chi2_sf({x}, {k}) gave {got:.5f}; the published table puts 5% of the mass above "
            f"{x}. Check the starting value for this parity of dof, and that each term divides "
            "by gamma(exponent + 1), not gamma(exponent)"
        )
    big = chi2_sf(400.0, 400)
    assert 0.4 < big < 0.6, (
        f"chi2_sf(400, 400) gave {big}; the median of a chi-square sits near its dof. Build "
        "each term through logarithms — powers and gammas overflow at this size"
    )
    assert chi2_sf(0.0, 3) == 1.0 and chi2_sf(-1.0, 3) == 1.0, "x <= 0 has upper tail 1.0"
    for bad in (0, 2.5, True):
        try:
            chi2_sf(1.0, bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"dof={bad!r} should raise ValueError")
    print("exercise 4 looks right — it reproduces the published table")

In [ ]:
_try("exercise 4", _check_chi2_sf)

With a tail you can have a critical value — the `x` whose tail is exactly `alpha` — by
bisection. It is given to you; it is only as right as your `chi2_sf`.

In [ ]:
def chi2_isf(alpha: float, dof: int) -> float:
    """The x with chi2_sf(x, dof) == alpha, by bisection on your chi2_sf. Given to you."""
    if not 0.0 < alpha < 1.0:
        raise ValueError(f"alpha must lie strictly between 0 and 1, got {alpha}")
    lo, hi = 0.0, 1.0
    while chi2_sf(hi, dof) > alpha:
        hi *= 2.0
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if chi2_sf(mid, dof) > alpha:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


def _show_critical_values() -> None:
    print("5% critical values from your chi2_sf, for comparison with any published table:")
    for k in (1, 2, 6, 8, 10):
        print(f"  {k:>2} degrees of freedom: {chi2_isf(ALPHA, k):.3f}")


_try("critical values", _show_critical_values, needs=("exercise 4",))

## 6. Exercise 5 — `hosmer_lemeshow()`

Hosmer and Lemeshow's statistic groups the records by predicted risk and applies a
chi-square test to the table of observed against expected events. In each group `j` of `n_j`
records with mean prediction `p̄_j`, compare the observed events `O_j` with the expected
`E_j = n_j · p̄_j`:

`C = Σ (O_j − E_j)² / (E_j · (1 − p̄_j))`

The groups are exercise 1's equal-frequency bins — the Stata manual documents exactly this
grouping, closed on the right, cut at the observation whose cumulative count first reaches
each share — so a tied score gives fewer groups than you asked for.

Then the part people get wrong. Hosmer and Lemeshow's 1980 abstract says the degrees of
freedom "depend on the particular statistic and the distributional assumptions". On the
sample the model was FITTED on, the statistic is approximately chi-square on `g − 2` —
the binary case of the `(g − 2) × (c − 1)` that Fagerland, Hosmer and Bofin give for its
multinomial form. On a sample outside the estimation sample the Stata manual's methods and
formulas give `g`: no parameter was estimated from those records. A validator's holdout is
outside. And `g` counts the groups FORMED, not requested: each formed group adds one squared,
standardised gap to the sum, and a group that was never formed adds nothing.

<details><summary>💡 Hint 1 — what to think about</summary>

Three decisions carry the marks. Where do the groups come from, and how many were there
really? What is in the denominator of each group's term — and what if a group's mean
prediction is 0 or 1? And where was the model fitted: on these records, or somewhere else?
Only the caller knows that, which is why the function refuses to guess.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate the arrays as exercise 2 does. Cut the groups with your exercise-1 function, place
the records exactly as exercise 2 does, and per group count the records, sum the outcomes and
sum the predictions. Refuse a group whose mean prediction is 0 or 1. Add up each group's
squared gap over its expected count times one minus its mean prediction. Degrees of freedom
are the groups formed, less two only when fitted_here is true; refuse fewer than one. The
p-value is your exercise-4 tail at the statistic.

</details>

In [ ]:
class HosmerLemeshow(NamedTuple):
    """The statistic, the reference distribution it was judged against, and the table."""

    statistic: float
    dof: int
    p_value: float
    groups: int              # groups actually formed, after coincident edges merged
    fitted_here: bool        # was the model fitted on this sample?
    count: np.ndarray        # records per group
    observed: np.ndarray     # events per group
    expected: np.ndarray     # sum of predictions per group


def hosmer_lemeshow(y_true: np.ndarray, y_prob: np.ndarray, n_groups: int = HL_GROUPS, *,
                    fitted_here: bool) -> HosmerLemeshow:
    """Hosmer-Lemeshow goodness-of-fit test over equal-frequency groups.

    `fitted_here` has no default, on purpose: whoever calls this must say whether the model's
    parameters were estimated on these very records.

    Requirements, each graded:
      * groups are `equal_frequency_edges(y_prob, n_groups)` with module 1's right-closed
        membership, and `groups` is how many that produced — possibly fewer than requested.
      * statistic = sum over groups of (O - E)^2 / (E * (1 - mean prediction)), where O is
        the events and E the sum of predictions in the group.
      * `dof = groups - 2` when `fitted_here` is True, `dof = groups` when it is False.
      * `p_value = chi2_sf(statistic, dof)`.
      * `ValueError` if the arrays are malformed (as in exercise 2), if a group's mean
        prediction is 0 or 1, or if fewer than one degree of freedom remains.
      * statistic and p_value are plain floats; dof and groups plain ints.

    Returns: a HosmerLemeshow record — statistic, dof, p_value, the groups formed, fitted_here,
    and the per-group count, observed and expected arrays.

    Example:
        >>> r = hosmer_lemeshow(np.array([0, 0, 0, 0, 1, 1, 1, 0, 0, 0]),
        ...                     np.array([0.1] * 5 + [0.5] * 5), 2, fitted_here=False)
        >>> round(r.statistic, 4), r.dof, r.groups, round(r.p_value, 4)
        (0.7556, 2, 2, 0.6854)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_hosmer_lemeshow() -> None:
    y = np.array([0, 0, 0, 0, 1, 1, 1, 0, 0, 0])
    p = np.array([0.1] * 5 + [0.5] * 5)
    r = hosmer_lemeshow(y, p, 2, fitted_here=False)
    assert isinstance(r, HosmerLemeshow), "return a HosmerLemeshow record"
    assert abs(r.statistic - 0.25 / 0.45 - 0.25 / 1.25) < 1e-12, (
        f"statistic {r.statistic:.6f}, expected 0.7556: group one has O=1, E=0.5, mean 0.1, "
        "so (1 - 0.5)^2 / (0.5 * 0.9); group two O=2, E=2.5, mean 0.5. Is (1 - mean) in the "
        "denominator?"
    )
    assert r.dof == 2 and r.groups == 2, (
        f"two groups on a sample the model was NOT fitted on have 2 degrees of freedom, got "
        f"{r.dof}. g - 2 is the rule for the fitting sample only"
    )
    assert abs(r.p_value - math.exp(-r.statistic / 2)) < 1e-9, (
        "p_value is chi2_sf(statistic, dof) — with 2 dof that is exp(-statistic / 2)"
    )
    tied = hosmer_lemeshow(np.array([0, 1] * 10), np.array([0.2] * 12 + [0.3] * 4 + [0.6] * 4),
                           10, fitted_here=False)
    assert tied.groups == 3 and tied.dof == 3, (
        f"ten groups requested over three distinct scores form {tied.groups} groups and "
        f"{tied.dof} dof; expected 3 and 3. The degrees of freedom follow the groups FORMED"
    )
    fitted = hosmer_lemeshow(np.array([0, 1] * 10), np.array([0.2] * 12 + [0.3] * 4 + [0.6] * 4),
                             10, fitted_here=True)
    assert fitted.dof == 1, (
        f"three groups on the fitting sample leave 3 - 2 = 1 dof, got {fitted.dof}"
    )
    try:
        hosmer_lemeshow(y, p, 2, fitted_here=True)
    except ValueError:
        pass
    else:
        raise AssertionError("two groups on the fitting sample leave 0 dof: ValueError")
    try:
        hosmer_lemeshow(y, p, 2)
    except TypeError:
        pass
    else:
        raise AssertionError("fitted_here must have no default: the caller has to say where "
                             "the model was fitted")
    print("exercise 5 looks right — and nobody can skip the degrees-of-freedom question")

In [ ]:
_try("exercise 5", _check_hosmer_lemeshow)

Here is the model on the validator's holdout, which it was not fitted on — then the same
statistic judged by the two wrong reference distributions people reach for.

In [ ]:
def _show_hl_on_holdout() -> None:
    right = hosmer_lemeshow(VAL["y"], VAL["pd"], HL_GROUPS, fitted_here=False)
    print(f"statistic {right.statistic:.4f} over {right.groups} groups "
          f"({HL_GROUPS} requested; the master scale's ties merged the rest)")
    readings = (("independent sample: g dof (right)", right.dof, right.p_value),
                ("fitting-sample rule, g - 2 (wrong here)", right.groups - 2,
                 chi2_sf(right.statistic, right.groups - 2)),
                ("requested groups, not formed (wrong)", HL_GROUPS,
                 chi2_sf(right.statistic, HL_GROUPS)))
    for name, dof, p in readings:
        verdict = "REJECTED" if p < ALPHA else "not rejected"
        print(f"  {name:<42} {dof:>2} dof   p {p:.4f}   {verdict} at {ALPHA:.2f}")
    print("\nOne statistic, three p-values. Only the first answers the question that was asked.")


_try("HL on the holdout", _show_hl_on_holdout, needs=("exercise 4", "exercise 5"))

Is `g` really right outside the fitting sample, and `g − 2` inside it? Do not take it on
trust — measure it. The cell below builds fresh samples with a known truth, twice:

- **fitted here:** a mis-scaled score is recalibrated by fitting two parameters on the
  sample itself, then tested on that same sample;
- **independent:** the exactly right probabilities are tested on a sample they were not
  fitted on — there was no fitting at all.

Neither model is wrong, so every rejection is a false alarm, and a correct reference
distribution rejects close to `ALPHA` of the time. `fit_logistic_2` is given to you; module 7
builds logistic regression properly.

In [ ]:
def fit_logistic_2(y: np.ndarray, x: np.ndarray, iterations: int = 50) -> np.ndarray:
    """Intercept and slope of a logistic regression of y on x, by Newton-Raphson. Given to you."""
    design = np.column_stack([np.ones_like(x), x])
    beta = np.zeros(2)
    for _ in range(iterations):
        p = _sigmoid(design @ beta)
        weight = p * (1.0 - p)
        step = np.linalg.solve(design.T @ (design * weight[:, None]), design.T @ (y - p))
        beta = beta + step
        if np.max(np.abs(step)) < 1e-10:
            break
    return beta


def _show_hl_dof_study() -> None:
    rng = np.random.default_rng(SEED + 5)
    n, g = 1500, HL_GROUPS
    rejections = {"fitted": [0, 0], "independent": [0, 0]}    # [under g - 2, under g]
    for _ in range(N_SIMS):
        x = rng.normal(0.0, 1.0, n)
        truth = _sigmoid(-2.2 + 0.9 * x)
        y = (rng.random(n) < truth).astype(np.int64)
        score = 1.5 * x + 0.3                                  # right shape, wrong scale
        beta = fit_logistic_2(y, score)
        fitted = hosmer_lemeshow(y, _sigmoid(beta[0] + beta[1] * score), g, fitted_here=True)
        indep = hosmer_lemeshow(y, truth, g, fitted_here=False)
        for key, r in (("fitted", fitted), ("independent", indep)):
            rejections[key][0] += chi2_sf(r.statistic, r.groups - 2) < ALPHA
            rejections[key][1] += chi2_sf(r.statistic, r.groups) < ALPHA
    print(f"{N_SIMS} samples of {n}, neither model wrong, test at {ALPHA:.2f}:")
    print(f"{'':<34}{'judged on g - 2':>16}{'judged on g':>13}")
    for key, label in (("fitted", "parameters fitted on the sample"),
                       ("independent", "model fitted elsewhere")):
        a, b = rejections[key]
        print(f"{label:<34}{a / N_SIMS:>16.1%}{b / N_SIMS:>13.1%}")
    print("\nWhat chi-square algebra predicts for the two mismatches, from your chi2_sf:")
    print(f"  independent sample judged on g - 2: "
          f"{chi2_sf(chi2_isf(ALPHA, g - 2), g):.1%} false rejections")
    print(f"  fitting sample judged on g:         "
          f"{chi2_sf(chi2_isf(ALPHA, g), g - 2):.1%} false rejections")


_try("degrees-of-freedom study", _show_hl_dof_study, needs=("exercise 4", "exercise 5"))

## 7. Stability, per driver — and where module 1's edges break

Module 1 measured PSI on the score. A score's PSI says the population moved; it cannot say
which input moved, and it can miss an input that moved a lot if the model barely listens to
it. A **characteristic stability index** — the industry's name for PSI computed on one input
at a time, against that input's own baseline — answers "which one".

Before you build it, try it with module 1's tools on the one input that changed in month A:
the share of applications arriving online.

In [ ]:
def _show_module1_edges_on_a_binary() -> None:
    base, cur = DEV["drivers"]["channel_online"], MON_A["drivers"]["channel_online"]
    edges = quantile_edges(base, N_BINS)
    result = population_stability_index(base, cur, edges)
    print(f"online share: development {base.mean():.1%}, month A {cur.mean():.1%}")
    print(f"module 1's quantile_edges on this yes/no input: {edges.tolist()}")
    print(f"that is {edges.size - 1} bin, and PSI over it is {result.psi:.4f}")
    print("\nOne bin holds everything, so both shares are 1.0 and the index is zero whatever")
    print("happens. Every quantile of a yes/no column is 0 or 1; de-duplicated, those are the")
    print("outer edges, and module 1 opens the outer edges to infinity.")


_show_module1_edges_on_a_binary()

## 8. Exercise 6 — `characteristic_edges()` and `characteristic_stability()`

**`characteristic_edges`** applies exercise 1's principle — edges sit on observed values,
coincident edges merge, no tie is split, no bin is empty — to module 1's STABILITY
convention, where bins are closed on the LEFT, `[e_i, e_i+1)`, and the outer edges are open
to infinity. Closed on the left, an edge must be the FIRST value of the next share rather
than the last value of this one, and an edge at the baseline's minimum would leave an empty
bin below it, so it goes.

**`characteristic_stability`** then runs module 1's PSI on each driver, with edges cut ONCE
on that driver's baseline, and adds the attribution: each driver's contribution to how far
the model's mean log-odds moved, `coefficient × (current mean − baseline mean)`. The model
is linear in its drivers on the log-odds scale, so those contributions add up EXACTLY to the
change in the mean log-odds. CSI says an input moved; the shift says how much of the score's
movement it caused.

<details><summary>💡 Hint 1 — what to think about</summary>

For the edges: under left-closed bins, which record's value should open the second share? On
data with no ties, your edges and exercise 1's should split the records identically. Then
think about a yes/no input: what should its bins be? For the index: which sample are the
edges cut on — and does that change from driver to driver?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Edges: validate, sort, and for k from 1 to n_bins minus 1 take the value at zero-based
position equal to the ceiling of k times n over n_bins, clipped to the last position.
De-duplicate, drop anything not strictly above the smallest value, and put minus and plus
infinity at the ends. Index: check the three mappings name the same drivers; for each driver
in sorted order cut edges on its baseline, run module 1's PSI, and multiply its coefficient by
the difference in means. Sort the rows by CSI, largest first, ties by name.

</details>

In [ ]:
class CharacteristicShift(NamedTuple):
    """One driver's stability, and its share of the score's movement."""

    driver: str
    csi: float                   # module 1's PSI on this driver, edges cut on its baseline
    bins: int                    # bins the driver's edges define
    log_odds_shift: float        # coefficient * (current mean - baseline mean)
    stability: StabilityResult   # the per-bin arithmetic, so the finding traces to a bin


def characteristic_edges(values: np.ndarray, n_bins: int = N_BINS) -> np.ndarray:
    """Stability bin edges for one input, cut on its baseline: ties never split, no bin empty.

    Requirements, each graded:
      * sort the values; for k = 1 .. n_bins - 1 the candidate edge is the value at ZERO-based
        position ceil(k * n / n_bins), clipped to n - 1. On untied data this splits the
        records exactly as exercise 1 does, under the opposite closure.
      * coincident candidates merge; a candidate not strictly above the smallest value is
        dropped (under left-closed bins it would leave an empty bin below it).
      * the result starts at -inf and ends at +inf, is strictly increasing, every interior
        edge is an observed value, and on the baseline it was cut from no bin
        (`[e_i, e_i+1)`, module 1's convention) is empty.
      * a yes/no input gets two bins. A constant input gets one — there is nothing to compare.
      * `ValueError` if `n_bins < 1`, if the input is empty, or if any value is not finite.

    Returns: the edges — a strictly increasing float array from -inf to +inf.

    Example:
        >>> characteristic_edges(np.array([0.0] * 75 + [1.0] * 25), 10).tolist()
        [-inf, 1.0, inf]
        >>> characteristic_edges(np.arange(10.0), 5).tolist()
        [-inf, 2.0, 4.0, 6.0, 8.0, inf]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def characteristic_stability(baseline: Mapping[str, np.ndarray],
                             current: Mapping[str, np.ndarray],
                             coefficients: Mapping[str, float],
                             n_bins: int = N_BINS) -> tuple:
    """Per-driver characteristic stability index, and each driver's log-odds shift.

    `baseline` and `current` map driver name -> values; `coefficients` maps driver name ->
    the model's log-odds coefficient (other keys, such as an intercept, are ignored).

    Requirements, each graded:
      * one row per driver in `baseline`; `csi` is module 1's `population_stability_index`
        of the current values against the baseline values, on `characteristic_edges` cut
        on the BASELINE values of that driver, with module 1's floor.
      * `bins` is how many bins those edges define; `stability` is the full StabilityResult.
      * `log_odds_shift = coefficient * (mean(current) - mean(baseline))` — positive when the
        driver pushed the model's log-odds up. Summed over drivers, the shifts equal the
        change in the model's mean log-odds.
      * rows ordered by `csi`, largest first; equal `csi` ordered by driver name.
      * csi and log_odds_shift plain floats, bins a plain int.
      * `ValueError` naming the drivers if `baseline` and `current` do not hold the same
        drivers, or if a driver has no coefficient.

    Returns: a tuple of CharacteristicShift, one per driver, largest CSI first.

    Example:
        >>> rows = characteristic_stability({"a": np.array([0.0, 1.0] * 50)},
        ...                                 {"a": np.array([1.0] * 100)}, {"a": 2.0})
        >>> rows[0].driver, rows[0].bins, rows[0].log_odds_shift
        ('a', 2, 1.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_characteristic_stability() -> None:
    binary = characteristic_edges(np.array([0.0] * 75 + [1.0] * 25), 10)
    assert binary.tolist() == [-np.inf, 1.0, np.inf], (
        f"a yes/no input gave edges {binary.tolist()}; it needs two bins, [-inf, 1, inf]. If "
        "you kept an edge at 0.0 you made an empty bin below the smallest value"
    )
    plain = characteristic_edges(np.arange(10.0), 5)
    assert plain.tolist() == [-np.inf, 2.0, 4.0, 6.0, 8.0, np.inf], (
        f"ten untied values in five bins gave {plain.tolist()}; each bin opens on the FIRST "
        "value of its share — zero-based position ceil(k * n / n_bins)"
    )
    base = {"flag": np.array([0.0, 1.0] * 50), "level": np.arange(100.0)}
    cur = {"flag": np.array([1.0] * 100), "level": np.arange(100.0)}
    rows = characteristic_stability(base, cur, {"flag": 2.0, "level": 0.5, "intercept": -1.0})
    assert [r.driver for r in rows] == ["flag", "level"], (
        "rows are ordered by CSI, largest first: 'flag' moved and 'level' did not"
    )
    want = population_stability_index(base["flag"], cur["flag"],
                                      characteristic_edges(base["flag"], N_BINS)).psi
    assert abs(rows[0].csi - want) < 1e-12, (
        f"CSI for 'flag' is {rows[0].csi:.6f}, expected {want:.6f} — cut the edges on the "
        "BASELINE values and run module 1's PSI, floor and all"
    )
    assert abs(rows[0].log_odds_shift - 1.0) < 1e-12, (
        f"'flag' moved its mean from 0.5 to 1.0 with coefficient 2.0: shift +1.0, got "
        f"{rows[0].log_odds_shift} — coefficient * (current mean - baseline mean)"
    )
    assert rows[1].csi == 0.0 and rows[1].bins == N_BINS, (
        "an unchanged input scores 0.0, over the bins its baseline supports"
    )
    try:
        characteristic_stability(base, {"flag": cur["flag"]}, {"flag": 2.0, "level": 0.5})
    except ValueError:
        pass
    else:
        raise AssertionError("a driver missing from the current sample should raise ValueError")
    print("exercise 6 looks right — each input's movement, and its share of the score's move")

In [ ]:
_try("exercise 6", _check_characteristic_stability)

Two monitoring months. Month A changed one input; month B changed another. Look at the
score's PSI first, as a monthly pack would, and then at the drivers.

In [ ]:
def _show_attribution() -> None:
    score_edges = characteristic_edges(DEV["pd"], N_BINS)
    for name, book in (("month A", MON_A), ("month B", MON_B)):
        score = population_stability_index(DEV["pd"], book["pd"], score_edges).psi
        rows = characteristic_stability(DEV["drivers"], book["drivers"], MODEL_LOG_ODDS)
        moved = float(book["log_odds"].mean() - DEV["log_odds"].mean())
        print(f"{name}: score PSI {score:.4f} over {score_edges.size - 1} bins")
        print(f"  {'driver':<16}{'CSI':>8}{'bins':>6}{'log-odds shift':>16}")
        for r in rows:
            print(f"  {r.driver:<16}{r.csi:>8.4f}{r.bins:>6}{r.log_odds_shift:>+16.4f}")
        total = sum(r.log_odds_shift for r in rows)
        print(f"  shifts sum to {total:+.4f}; the model's mean log-odds moved {moved:+.4f}\n")
    print("A driver can move a long way and barely move the score, if the model gives it little")
    print("weight. The score's PSI reports the effect; only the drivers report the cause.")


_try("attribution", _show_attribution, needs=("exercise 6",))

## 9. Exercise 7 — `psi_null_bootstrap()`

Every PSI above was compared, implicitly, with zero. It is never zero. Two samples drawn from
the SAME population disagree bin by bin, and PSI adds those disagreements up. So the useful
question about a threshold is: **if nothing at all had drifted, how often would it fire?**

Bootstrap the whole monitoring procedure under no drift. Treat the development sample as the
population. In each replicate, draw a development sample and a monitoring sample from it at
the sizes you actually have, cut the edges ONCE on the replicate's development sample — as
production does — and score the monitoring sample against them. The spread of those PSIs is
the null distribution at your sample sizes.

Yurdakul and Naranjo showed that under no shift, PSI is approximately `(1/n + 1/m)` times a
chi-square on `B − 1` degrees of freedom, for a baseline of `n`, a current sample of `m` and
`B` bins. Both terms matter: the baseline is a sample too.

<details><summary>💡 Hint 1 — what to think about</summary>

Two samples per replicate, and which one are the edges cut on? The monitoring month is not
compared with the population; it is compared with a development sample that was itself
drawn. Which size goes with which draw? And where must every random number come from, if
the result is to be reproduced?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate: a non-empty, finite population; sizes and replicate count of at least one; a numpy
Generator. For each replicate, draw the development sample's indices and then the monitoring
sample's, with replacement, from the generator's integers method; cut the edges on the drawn
development values with your exercise-6 edge function; and store module 1's PSI of the drawn
monitoring values against them. Return the replicates as a float array.

</details>

In [ ]:
def psi_null_bootstrap(population: np.ndarray, n_baseline: int, n_current: int,
                       rng: np.random.Generator, n_bins: int = N_BINS,
                       n_boot: int = N_BOOT) -> np.ndarray:
    """PSI under no drift at all, at the sizes you actually have. `n_boot` replicates.

    Requirements, each graded:
      * each replicate draws a baseline of `n_baseline` and then a current sample of
        `n_current`, both WITH replacement from `population`, both from `rng` (use
        `rng.integers(0, len(population), size)` for the indices).
      * the replicate's edges are `characteristic_edges(replicate_baseline, n_bins)` — cut
        once, on the replicate's baseline, never on the current sample and never on the
        whole population.
      * the replicate's value is module 1's `population_stability_index(...).psi`, floored.
      * returns a float array of length `n_boot`; the same seed gives the same array.
      * `ValueError` if the population is empty or not finite, if any size or `n_boot` is
        below 1, or if `rng` is not a `numpy.random.Generator`.

    Returns: a float array of `n_boot` PSI values, one per no-drift replicate.

    Example:
        >>> z = psi_null_bootstrap(np.arange(1000.0), 400, 400, np.random.default_rng(0),
        ...                        n_boot=200)
        >>> z.shape, bool(z.min() >= 0.0)
        ((200,), True)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_psi_null_bootstrap() -> None:
    pop = np.arange(1000.0)
    z = psi_null_bootstrap(pop, 400, 400, np.random.default_rng(1), n_boot=200)
    assert isinstance(z, np.ndarray) and z.shape == (200,), (
        f"return an array of n_boot values, got {type(z).__name__} of shape {np.shape(z)}"
    )
    assert np.all(np.isfinite(z)) and z.min() >= 0.0, "every replicate PSI is finite and >= 0"
    again = psi_null_bootstrap(pop, 400, 400, np.random.default_rng(1), n_boot=200)
    assert np.array_equal(z, again), (
        "the same seed must give the same replicates — take every draw from `rng`"
    )
    expect = (N_BINS - 1) * (1 / 400 + 1 / 400)
    assert abs(z.mean() - expect) < 0.2 * expect, (
        f"mean null PSI {z.mean():.4f}, while chi-square theory puts it near {expect:.4f}. "
        "About half that means the baseline never varied — a fresh baseline must be drawn, "
        "with replacement, in EVERY replicate. Exactly zero means one index draw served both "
        "samples"
    )
    flags = psi_null_bootstrap(np.array([0.0] * 700 + [1.0] * 300), 400, 400,
                               np.random.default_rng(2), n_boot=100)
    assert flags.max() > 0.0, (
        "on a yes/no population every replicate scored 0.0 — the edges collapsed to one bin. "
        "Cut them with characteristic_edges, not module 1's quantile_edges"
    )
    print("exercise 7 looks right — PSI's null, at the sizes you choose")

In [ ]:
_try("exercise 7", _check_psi_null_bootstrap)

At your own sizes — the development sample against one monitoring month — here is what
the model's score PSI reads when nothing has happened.

In [ ]:
def _show_null_at_your_size() -> None:
    bins = characteristic_edges(DEV["pd"], N_BINS).size - 1
    z = psi_null_bootstrap(DEV["pd"], N_DEV, N_MON, np.random.default_rng(SEED + 7))
    scale = 1.0 / N_DEV + 1.0 / N_MON
    print(f"score PSI under no drift, {N_DEV} against {N_MON} records, {bins} bins, "
          f"{z.size} replicates:")
    print(f"  mean {z.mean():.4f}   95th percentile {np.quantile(z, 0.95):.4f}   "
          f"largest {z.max():.4f}")
    print(f"  chi-square approximation: mean {scale * (bins - 1):.4f}, 95th percentile "
          f"{scale * chi2_isf(0.05, bins - 1):.4f}")


_try("null at your size", _show_null_at_your_size, needs=("exercise 4", "exercise 6", "exercise 7"))

## 10. Exercise 8 — `false_alarm_rate()` and `threshold_for_rate()`

A null distribution turns a threshold into a number with a cost. **`false_alarm_rate`** is
the share of undrifted replicates that would breach it — with module 1's breach rule, strictly
greater than, so a PSI exactly on the threshold is within it — and the Monte Carlo standard
error of that share, because it is itself an estimate from `n_boot` draws.

**`threshold_for_rate`** runs the other way: given the false-alarm rate you are prepared to
pay, it returns the smallest replicate value that at most that share of the null exceeds.
The industry's 0.10 and 0.25 bands are conventions; Yurdakul and Naranjo note they are used
"without reference to statistical type I or type II error rates". This function is the
reference.

<details><summary>💡 Hint 1 — what to think about</summary>

Is a replicate that lands exactly on the threshold an alarm? Module 1 answered that. For the
inverse: how many replicates are ALLOWED above the threshold, and which sorted replicate
leaves exactly that many above it? Watch what floating-point does to rate times n_boot.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Rate: count the replicates strictly above the threshold and divide by how many there are; the
standard error is the square root of rate times one minus rate over that count. Inverse: sort
the replicates; the alarms allowed are the floor of rate times the count, nudged by a tiny
tolerance so a product that should be whole is; the threshold is the value that many places
below the top of the sorted list. Validate both: a non-empty, finite null, and a rate in
[0, 1).

</details>

In [ ]:
class AlarmRate(NamedTuple):
    """How often a threshold fires on data that has not drifted."""

    threshold: float
    rate: float        # share of null replicates strictly above the threshold
    se: float          # Monte Carlo standard error of that share
    n_boot: int


def false_alarm_rate(null_psi: np.ndarray, threshold: float) -> AlarmRate:
    """The false-alarm rate of `threshold` against a null distribution of PSI.

    Requirements, each graded:
      * `rate` is the share of `null_psi` STRICTLY above `threshold` — module 1's breach rule,
        so a value exactly on the threshold is not an alarm.
      * `se = sqrt(rate * (1 - rate) / n_boot)`; `n_boot` is the number of replicates.
      * floats are plain Python floats, `n_boot` a plain int.
      * `ValueError` if `null_psi` is empty or holds a value that is not finite.

    Returns: an AlarmRate — the threshold, the rate, its standard error and n_boot.

    Example:
        >>> false_alarm_rate(np.array([0.0, 0.1, 0.1, 0.2, 0.3]), 0.1).rate
        0.4
    """
    # YOUR CODE HERE
    raise NotImplementedError


def threshold_for_rate(null_psi: np.ndarray, rate: float) -> float:
    """The smallest null value that at most `rate` of the null distribution exceeds.

    Requirements, each graded:
      * the result is one of the values in `null_psi`, and the share of `null_psi` strictly
        above it is at most `rate`; no smaller value in `null_psi` has that property.
      * the alarms allowed are `floor(rate * n_boot)`, computed with a tolerance of about
        1e-9 so that a product that should be a whole number is one.
      * `rate = 0` returns the largest value; the result is a plain float.
      * `ValueError` if `null_psi` is empty or not finite, or `rate` is outside [0, 1).

    Returns: the threshold, a plain Python float that occurs in `null_psi`.

    Example:
        >>> threshold_for_rate(np.array([0.0, 0.1, 0.1, 0.2, 0.3]), 0.2)
        0.2
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_false_alarms() -> None:
    null = np.array([0.0, 0.1, 0.1, 0.2, 0.3])
    r = false_alarm_rate(null, 0.1)
    assert isinstance(r, AlarmRate), "return an AlarmRate record"
    assert abs(r.rate - 0.4) < 1e-12, (
        f"two of five replicates are strictly above 0.1, a rate of 0.4; got {r.rate}. 0.8 "
        "means you counted the two sitting exactly ON the threshold as alarms"
    )
    assert abs(r.se - math.sqrt(0.4 * 0.6 / 5)) < 1e-12, "se = sqrt(rate * (1 - rate) / n_boot)"
    assert r.n_boot == 5, "n_boot is the number of replicates"
    t = threshold_for_rate(null, 0.2)
    assert t == 0.2, (
        f"at most one of five may lie above the threshold: 0.2 leaves one above it, and 0.1 "
        f"leaves two. Got {t}"
    )
    assert threshold_for_rate(null, 0.0) == 0.3, "a rate of 0 allows no alarm: the largest value"
    many = np.arange(1000.0)
    assert threshold_for_rate(many, 0.05) == 949.0, (
        "on 0..999 at 5%, fifty values may lie above: the threshold is 949. 948 means "
        "0.05 * 1000 came out as 49.999... — floor it with a tolerance"
    )
    try:
        threshold_for_rate(null, 1.0)
    except ValueError:
        pass
    else:
        raise AssertionError("a rate of 1.0 allows every replicate to alarm — ValueError")
    print("exercise 8 looks right — a threshold now comes with its price")

In [ ]:
_try("exercise 8", _check_false_alarms)

The same two conventional thresholds, at the monitoring sizes a validator actually meets: a
full month, a quarter's worth of one product, a single segment. Nothing in any of these
samples has drifted.

In [ ]:
def _show_false_alarm_table() -> None:
    bins = characteristic_edges(DEV["pd"], N_BINS).size - 1
    rng = np.random.default_rng(SEED + 9)
    header = "".join(f"{f'FAR at {t:.2f}':>16}{'chi-square':>12}" for t in PSI_THRESHOLDS)
    print(f"{'records':>8}{header}{'5% threshold':>14}")
    for m in SIZES:
        z = psi_null_bootstrap(DEV["pd"], N_DEV, m, rng)
        scale = 1.0 / N_DEV + 1.0 / m
        cells = ""
        for t in PSI_THRESHOLDS:
            r = false_alarm_rate(z, t)
            cells += f"{f'{r.rate:.1%} ± {r.se:.1%}':>16}{chi2_sf(t / scale, bins - 1):>12.1%}"
        print(f"{m:>8}{cells}{threshold_for_rate(z, 0.05):>14.4f}")
    print(f"\n(development sample {N_DEV} records, {bins} bins, {N_BOOT} replicates per row)")
    print("The same threshold is almost silent at one size and cries wolf at another. A threshold")
    print("that does not say what size it was set for has not said what it costs.")


_try("false-alarm table", _show_false_alarm_table,
     needs=("exercise 4", "exercise 6", "exercise 7", "exercise 8"))

## 11. Exercise 9 — `render_appendix()`

Everything above produced evidence. This produces the calibration-and-stability appendix
of the validation report — from that evidence and nothing else, as module 1's report was.
Its one new rule is the point of the lesson: **no threshold appears without its
false-alarm rate.** A reader who sees "PSI 0.12 against 0.10: BREACH" must also see how often
that threshold fires at this size when nothing has happened.

<details><summary>💡 Hint 1 — what to think about</summary>

This function formats; it does not measure. Every figure comes out of `findings` at the
precision the docstring sets. Count the threshold lines you will write — one for the ECE
ceiling, one for the goodness-of-fit test, and one per threshold for the score and for every
driver in every month — and make sure each carries its rate. The verdict boundary is module
1's: exactly on a threshold is within it.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Check the top-level keys and each month's keys, naming everything missing in one ValueError.
Build a list of lines: the title, then each heading from APPENDIX_SECTIONS with its lines —
the data note and sizes; the sensitivity table row by row, the range, and the reported-ECE
threshold line; the goodness-of-fit sentence and its threshold line; then, per month, its
sub-heading, a score line per threshold and a driver line per driver per threshold. Join
with newlines.

</details>

In [ ]:
APPENDIX_SECTIONS = ("## A.1 Data", "## A.2 Calibration and the binning choice",
                     "## A.3 Goodness of fit", "## A.4 Population stability")
APPENDIX_KEYS = ("model_id", "data_note", "n_calibration", "n_baseline", "ece_rows",
                 "ece_reported", "ece_ceiling", "ece_far", "hl", "hl_requested", "hl_alpha",
                 "hl_far", "thresholds", "months")
MONTH_KEYS = ("name", "n_current", "score_psi", "score_bins", "score_far", "shifts", "csi_far")


def render_appendix(findings: Mapping[str, Any]) -> str:
    """Render the calibration-and-stability appendix as markdown, entirely from `findings`.

    `findings` carries every key in APPENDIX_KEYS. `months` is a list of dicts, each carrying
    every key in MONTH_KEYS: `score_far` maps threshold -> rate, and `csi_far` maps driver ->
    {threshold -> rate}. Requirements, each graded:
      * line 1 is "# Calibration and stability appendix — <model_id>", and the four headings in
        APPENDIX_SECTIONS follow in order.
      * A.1: the data note, then "Calibration sample: <n_calibration> records. Stability
        baseline: <n_baseline> records."
      * A.2: a markdown table, header "| scheme | bins requested | bins used | smallest bin |
        ECE |", one row per SensitivityRow as "| <scheme> | <requested> | <bins_used> |
        <min_support> | <ece:.4f> |"; then "ECE across these <n> schemes runs from <lo:.4f>
        to <hi:.4f>."; then the threshold line for `ece_reported`.
      * A.3: "Hosmer-Lemeshow statistic <stat:.4f> over <groups> groups (<hl_requested>
        requested), <dof> degrees of freedom (<basis>), p-value <p:.4f>." where basis is
        "fitted on this sample" or "independent of the fit"; then its threshold line, whose
        verdict is REJECTED when p < hl_alpha and NOT REJECTED otherwise.
      * A.4: per month, "### <name>: <n_current> records against <n_baseline>", then one line
        per threshold for the score, then one line per threshold for each shift in the order
        given.
      * EVERY threshold line ends "(false-alarm rate <rate:.1%> at this size)". The verdicts
        are BREACH when the value is strictly above the threshold and WITHIN otherwise. The
        exact line formats:
          "- reported ECE <ece:.4f> (<scheme>, <requested> bins requested, <bins_used> used)
             against <ceiling:.2f>: <verdict> (false-alarm rate ...)"
          "- p-value <p:.4f> against <alpha:.2f>: <verdict> (false-alarm rate ...)"
          "- score PSI <psi:.4f> over <score_bins> bins against <t:.2f>: <verdict> (...)"
          "- CSI <driver> <csi:.4f> over <bins> bins, log-odds shift <shift:+.4f>, against
             <t:.2f>: <verdict> (false-alarm rate ...)"
      * nothing is recomputed: every figure is read from `findings`.
      * `ValueError` naming every missing key, at the top level or in any month.

    Returns: the appendix, one markdown string with its lines joined by newlines.

    Example:
        >>> render_appendix(findings).splitlines()[0]          # doctest: +SKIP
        '# Calibration and stability appendix — scorecard-v7'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _probe_findings() -> dict:
    """A small findings dict with unmistakable numbers, so you can see where each one lands."""
    rows = (SensitivityRow("equal-width", 5, 5, 3, 77, 0.0111),
            SensitivityRow("equal-frequency", 10, 7, 7, 302, 0.0333))
    stab = StabilityResult(psi=0.4444, contributions=np.array([0.4444]),
                           expected_pct=np.array([1.0]), actual_pct=np.array([1.0]))
    shift = CharacteristicShift(driver="probe_driver", csi=0.4444, bins=2,
                                log_odds_shift=0.0123, stability=stab)
    hl = HosmerLemeshow(statistic=12.3456, dof=7, p_value=0.0898, groups=7, fitted_here=False,
                        count=np.array([1]), observed=np.array([0.0]), expected=np.array([0.5]))
    month = {"name": "probe month", "n_current": 321, "score_psi": 0.25, "score_bins": 6,
             "score_far": {0.10: 0.0555, 0.25: 0.0066}, "shifts": (shift,),
             "csi_far": {"probe_driver": {0.10: 0.0777, 0.25: 0.0088}}}
    return {"model_id": "probe-model", "data_note": "PROBE NOTE", "n_calibration": 1234,
            "n_baseline": 5678, "ece_rows": rows, "ece_reported": rows[1], "ece_ceiling": 0.04,
            "ece_far": 0.0222, "hl": hl, "hl_requested": 10, "hl_alpha": 0.05,
            "hl_far": 0.0444, "thresholds": (0.10, 0.25), "months": [month]}


def _check_render_appendix() -> None:
    probe = _probe_findings()
    text = render_appendix(probe)
    assert text.splitlines()[0] == "# Calibration and stability appendix — probe-model", (
        f"first line was {text.splitlines()[0]!r}; it names the model from findings"
    )
    positions = [text.find(h) for h in APPENDIX_SECTIONS]
    assert all(pos >= 0 for pos in positions) and positions == sorted(positions), (
        "the four APPENDIX_SECTIONS headings must all appear, in order"
    )
    assert "| equal-frequency | 10 | 7 | 302 | 0.0333 |" in text, (
        "each sensitivity row prints as '| scheme | requested | used | smallest | ECE |'"
    )
    assert "7 degrees of freedom (independent of the fit)" in text, (
        "the goodness-of-fit sentence states the dof it used and why"
    )
    assert "- score PSI 0.2500 over 6 bins against 0.25: WITHIN (false-alarm rate 0.7% at this " \
           "size)" in text, (
        "a PSI of exactly 0.25 against 0.25 is WITHIN — breach means strictly above — and the "
        "line carries the threshold's false-alarm rate from findings"
    )
    thresholds = 2 + len(probe["thresholds"]) * (1 + 1)
    assert text.count("(false-alarm rate ") == thresholds, (
        f"{text.count('(false-alarm rate ')} threshold lines carry a false-alarm rate; this "
        f"probe has {thresholds} thresholds and every one of them must"
    )
    try:
        render_appendix({k: v for k, v in probe.items() if k != "hl_far"})
    except ValueError as exc:
        assert "hl_far" in str(exc), f"the error should name the missing key, got {exc}"
    else:
        raise AssertionError("incomplete findings should raise ValueError")
    print("exercise 9 looks right — no threshold leaves without its price")

In [ ]:
_try("exercise 9", _check_render_appendix)

## 12. Generate the appendix

Nine functions, one document. The cell below measures every false-alarm rate the appendix
quotes — for the ECE ceiling, by drawing outcomes from the model's own predictions; for the
goodness-of-fit test, the same way; for every PSI and CSI threshold, from exercise 7's
bootstrap on that input — and hands the lot to your renderer. Nothing below types a number.

In [ ]:
def run_appendix() -> dict:
    """Assemble every finding and return the appendix with them."""
    rng = np.random.default_rng(SEED + 12)
    rows = ece_sensitivity(VAL["y"], VAL["pd"], BIN_COUNTS)
    reported = next(r for r in rows if r.scheme == "equal-frequency" and r.requested == N_BINS)
    ece_null = exact_calibration_ece(VAL["pd"], equal_frequency_edges(VAL["pd"], N_BINS),
                                     N_ECE_SIMS, rng)
    hl = hosmer_lemeshow(VAL["y"], VAL["pd"], HL_GROUPS, fitted_here=False)
    hl_alarms = 0
    for _ in range(N_SIMS):
        simulated = (rng.random(N_VAL) < VAL["pd"]).astype(np.int64)
        hl_alarms += hosmer_lemeshow(simulated, VAL["pd"], HL_GROUPS,
                                     fitted_here=False).p_value < ALPHA

    nulls: dict = {}

    def rates(name: str, population: np.ndarray, n_current: int) -> dict:
        if (name, n_current) not in nulls:
            z = psi_null_bootstrap(population, N_DEV, n_current, rng)
            nulls[(name, n_current)] = {t: false_alarm_rate(z, t).rate for t in PSI_THRESHOLDS}
        return nulls[(name, n_current)]

    score_edges = characteristic_edges(DEV["pd"], N_BINS)
    months = []
    for name, book in (("Month A (the online month)", MON_A),
                       ("Month B (the utilisation month)", MON_B)):
        n_current = book["y"].size
        shifts = characteristic_stability(DEV["drivers"], book["drivers"], MODEL_LOG_ODDS)
        months.append({
            "name": name, "n_current": n_current,
            "score_psi": population_stability_index(DEV["pd"], book["pd"], score_edges).psi,
            "score_bins": score_edges.size - 1,
            "score_far": rates("score", DEV["pd"], n_current),
            "shifts": shifts,
            "csi_far": {s.driver: rates(s.driver, DEV["drivers"][s.driver], n_current)
                        for s in shifts},
        })
    findings = {
        "model_id": "retail scorecard v7, master-scale PD",
        "data_note": DATA_NOTE,
        "n_calibration": N_VAL, "n_baseline": N_DEV,
        "ece_rows": rows, "ece_reported": reported, "ece_ceiling": ECE_CEILING,
        "ece_far": false_alarm_rate(ece_null, ECE_CEILING).rate,
        "hl": hl, "hl_requested": HL_GROUPS, "hl_alpha": ALPHA, "hl_far": hl_alarms / N_SIMS,
        "thresholds": PSI_THRESHOLDS, "months": months,
    }
    return {"findings": findings, "appendix": render_appendix(findings)}


def _show_appendix() -> None:
    out = run_appendix()
    print(out["appendix"])
    assert out["appendix"].count("(false-alarm rate ") >= 4, "every threshold carries its rate"


_try("the appendix", _show_appendix, needs=tuple(_EXERCISES))

## 13. Common mistakes

- **Quoting an ECE without its scheme.** The table in section 4 is the reason. Name the
  scheme, the bins requested and the bins used, or the number cannot be reproduced.
- **Splitting ties to force equal-frequency bins.** Two identical scores in two bins is a
  result that depends on sort order. Merge, and report the bins you got.
- **Hosmer-Lemeshow on `g − 2` for a holdout.** That is the fitting-sample rule. On a sample
  the model never saw it inflates false rejections — section 6 measured by how much.
- **Degrees of freedom from the groups requested.** A tied score forms fewer groups. The
  test is on the groups it formed.
- **One CSI binning for every input.** Module 1's quantile edges put a yes/no input in one
  bin and report zero forever. Cut each input's edges on its own baseline, ties intact.
- **Reading a flat score PSI as a stable population.** A heavily weighted input and a
  lightly weighted one can move by the same amount and move the score by very different
  amounts. The drivers are where the cause is.
- **One index draw for both samples.** The replicate's development sample and its month are
  then the same records, every replicate scores zero, and every threshold looks free.
- **A null that holds the baseline fixed.** The development sample was drawn too. Freeze it
  and the null comes out too narrow, so a threshold set on it fires more than it promised.
- **One threshold for every sample size.** The false-alarm table is the reason.

The frozen baseline is worth seeing, because it looks like a perfectly good bootstrap. The
cell below sets a 5% threshold on a null that draws only the month, then measures how often
that threshold fires on the honest null, where the development sample is drawn too.

In [ ]:
def _show_frozen_null() -> None:
    rng = np.random.default_rng(SEED + 13)
    honest = psi_null_bootstrap(DEV["pd"], N_DEV, N_MON, rng)
    edges = characteristic_edges(DEV["pd"], N_BINS)
    frozen = np.array([population_stability_index(
        DEV["pd"], DEV["pd"][rng.integers(0, N_DEV, N_MON)], edges).psi for _ in range(N_BOOT)])
    promised = threshold_for_rate(frozen, 0.05)
    print(f"{N_DEV} development records against a month of {N_MON}, {N_BOOT} replicates each:")
    print(f"  null with the development sample frozen:  mean {frozen.mean():.4f}")
    print(f"  null with the development sample drawn:   mean {honest.mean():.4f}")
    print(f"\nthe threshold the frozen null sets for a 5% false-alarm rate: {promised:.4f}")
    print(f"how often it fires when nothing has drifted: "
          f"{false_alarm_rate(honest, promised).rate:.1%}")
    print("\nThe baseline in production is one sample among many that the development period could")
    print("have produced. A null that forgets that promises a rate it does not deliver.")


_try("frozen baseline", _show_frozen_null, needs=("exercise 6", "exercise 7", "exercise 8"))

## 14. Self-check

1. A holdout's ECE reads 0.011 with five equal-width bins and 0.027 with twenty
   equal-frequency bins, on the same records. The right reading is:
   - (a) the equal-frequency figure is biased upwards, so 0.011 is the ECE
   - (b) coarse bins merged grades whose errors cancel; neither figure is "the" ECE, and the
         report must name the scheme and the bins used
   - (c) the model's calibration changed between the two runs

2. You ask for twenty-five equal-frequency bins on a master-scale PD and get seven. This is:
   - (a) a bug; equal-frequency binning always returns the bins requested
   - (b) fixable by jittering the scores so the ties break
   - (c) the right answer: the grades' ties merge coincident edges, and the report states
         seven

3. A validator runs Hosmer-Lemeshow on a holdout the model never saw: seven groups formed,
   statistic 13.1. The p-value comes from a chi-square on:
   - (a) 7 degrees of freedom
   - (b) 5 degrees of freedom
   - (c) 10 degrees of freedom, because 10 groups were requested

4. A segment of 80 records scores PSI 0.12 against the 0.10 band. Before calling it drift you
   need:
   - (a) a second month's PSI above 0.10
   - (b) nothing more; 0.12 is above 0.10
   - (c) the false-alarm rate of 0.10 at 80 records against your baseline, from the null

5. In one month the score's PSI sits well inside both bands, while the CSI of
   `channel_online` is far above the upper one. The right reading is:
   - (a) the model is broken, because an input has breached
   - (b) the input moved a long way and the model gives it little weight, so the score barely
         moved; the shift is still a finding about the population the model now serves
   - (c) CSI is unreliable on a yes/no input, so the score PSI is the one to believe

Answers are in this lesson's worked solution in the course repository.

In [ ]:
print(f"\nlesson wall time: {time.perf_counter() - _LESSON_T0:.1f}s")

## What you built, and where it goes next

Nine functions and a generated appendix: calibration bins that respect ties, the table
that shows how far one model's ECE moves with the binning, a chi-square tail built from
scratch and a goodness-of-fit test that asks where the model was fitted, a stability index
that names the input that moved, and a null distribution that puts a price on every
threshold. Module 9 of this programme runs module 1's PSI over a file too large for memory;
the null distribution you built here is what tells it where to set the alarm. Module 10
folds this appendix into the committee pack.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<11} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_equal_frequency_edges),
                              ("exercise 2", _check_ece_on_edges),
                              ("exercise 3", _check_ece_sensitivity),
                              ("exercise 4", _check_chi2_sf),
                              ("exercise 5", _check_hosmer_lemeshow),
                              ("exercise 6", _check_characteristic_stability),
                              ("exercise 7", _check_psi_null_bootstrap),
                              ("exercise 8", _check_false_alarms),
                              ("exercise 9", _check_render_appendix)):
            _try(_name, _check)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))